In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

DOCS_PD = pd.DataFrame({"document_id": [101, 102, 103], "title": ["A", "B", "C"], "semanticscholar_url": ["https://api.semanticscholar.org/graph/v1/paper/a", "https://api.semanticscholar.org/graph/v1/paper/b", "https://api.semanticscholar.org/graph/v1/paper/c"]}).set_index("document_id")
DOCS_PD.index = pd.Index(pd.array(DOCS_PD.index, dtype="Int64"), name="document_id")
DOCS_PL = pl.DataFrame({"d3_document_id": [101, 102, 103], "title": ["A", "B", "C"], "semanticscholar_url": ["https://api.semanticscholar.org/graph/v1/paper/a", "https://api.semanticscholar.org/graph/v1/paper/b", "https://api.semanticscholar.org/graph/v1/paper/c"]})

def _set_docs_pd():
    global self
    self = SimpleNamespace(documents_data=DOCS_PD.copy())

def _set_docs_pl():
    global self
    self = SimpleNamespace(documents_data=DOCS_PL.clone())

# --- input_loc_col ---
FIX_INPUT_LOC_COL_D3_DOCUMENT_ID = 101

# --- input_loc_index_item ---
FIX_INPUT_LOC_INDEX_ITEM_SEMANTICSCHOLAR_URL = "https://api.semanticscholar.org/graph/v1/paper/a"

_set_docs_pd()
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_input_loc_col(d3_document_id):
    return cast(str, self.documents_data.loc[d3_document_id, "semanticscholar_url"])
    return None

def before_input_loc_index_item(semanticscholar_url):
    return (
        self.documents_data.loc[
            self.documents_data["semanticscholar_url"] == semanticscholar_url
        ]
        .index.item().item()
    )


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_input_loc_col(d3_document_id):
    from typing import cast

    return cast(
        str,
        self.documents_data.filter(pl.col("d3_document_id") == d3_document_id)
        .select("semanticscholar_url")
        .item(),
    )

def gen_input_loc_index_item(semanticscholar_url):

    self.documents_data.filter(
        pl.col("semanticscholar_url") == semanticscholar_url
    ).row(0)[0]
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: input_loc_col ===

try:
    _set_docs_pl()
    _r = gen_input_loc_col(FIX_INPUT_LOC_COL_D3_DOCUMENT_ID)
    print("✅ L1 smoke gen_input_loc_col: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_input_loc_col: {type(_e).__name__}: {_e}")

try:
    _set_docs_pd()
    _rb = before_input_loc_col(FIX_INPUT_LOC_COL_D3_DOCUMENT_ID)
    print("✅ L1 smoke before_input_loc_col: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_input_loc_col: {type(_e).__name__}: {_e}")

try:
    _set_docs_pd()
    _rb = before_input_loc_col(FIX_INPUT_LOC_COL_D3_DOCUMENT_ID)
    _set_docs_pl()
    _rg = gen_input_loc_col(FIX_INPUT_LOC_COL_D3_DOCUMENT_ID)
    if _rb == _rg:
        print("✅ L2 equivalence input_loc_col: MATCH")
    else:
        print(f"❌ L2 equivalence input_loc_col: MISMATCH - before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence input_loc_col: setup error - {type(_e).__name__}: {_e}")

try:
    _before_err = _gen_err = None
    try:
        _set_docs_pd()
        before_input_loc_col(999)
    except Exception as _e:
        _before_err = _e
    try:
        _set_docs_pl()
        gen_input_loc_col(999)
    except Exception as _e:
        _gen_err = _e
    if _before_err is not None and _gen_err is not None and not isinstance(_gen_err, (SyntaxError, NameError)):
        print(f"✅ L3 edge input_loc_col missing id: MATCH - both rejected (before={type(_before_err).__name__}, gen={type(_gen_err).__name__})")
    else:
        print(
            "❌ L3 edge input_loc_col missing id: MISMATCH - "
            f"before_error={type(_before_err).__name__ if _before_err else None}, "
            f"gen_error={type(_gen_err).__name__ if _gen_err else None}"
        )
except Exception as _e:
    print(f"❌ L3 edge input_loc_col missing id: setup error - {type(_e).__name__}: {_e}")
